# Definition of the files used for comparison

In [27]:
# The inference (or preprocessing) environment works for this notebook
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

import pandas as pd

# Path to the Ground Truth reference .h5 file (*_pwrflw.h5)
path_to_Reference = "/home/pedro/Linux-AntigravityProjects/SurroGrid-linux/Inference/1.Cheap_Ground_Truth/gridalloc/results/0197_N2780244E4431460_80689_2_2_PV100_HP100_EV100_VarTar0_CapPr0_pwrflw.h5"

# Print the number of buildings in this grid and list available tables
with pd.HDFStore(path_to_Reference, mode="r") as store:
    df_bldgs_ref = store["/raw_data/buildings"]
    n_res = int((df_bldgs_ref["nonresidential_floor_area"] == 0).sum())
    n_nonres = int((df_bldgs_ref["residential_floor_area"] == 0).sum())
    n_mixed = int(((df_bldgs_ref["residential_floor_area"] != 0) & (df_bldgs_ref["nonresidential_floor_area"] != 0)).sum())
    print(f"Reference grid buildings: {len(df_bldgs_ref)} total ({n_res} pure res, {n_nonres} pure non-res, {n_mixed} mixed)")

    keys = store.keys()
    print("Available tables in Reference file:")
    for k in keys:
        print(f" - {k}")


Reference grid buildings: 125 total (118 pure res, 6 pure non-res, 1 mixed)
Available tables in Reference file:
 - /performance
 - /urbs_out/tsam/kept_timesteps
 - /urbs_out/reduced_data/buy_sell_price
 - /urbs_out/reduced_data/commodity
 - /urbs_out/reduced_data/demand
 - /urbs_out/reduced_data/eff_factor
 - /urbs_out/reduced_data/global_prop
 - /urbs_out/reduced_data/process
 - /urbs_out/reduced_data/process_commodity
 - /urbs_out/reduced_data/site
 - /urbs_out/reduced_data/storage
 - /urbs_out/reduced_data/supim
 - /urbs_out/reduced_data/type_period
 - /urbs_out/reduced_data/weather
 - /urbs_out/MILP/cap_pro
 - /urbs_out/MILP/cap_pro_new
 - /urbs_out/MILP/cap_sto_c
 - /urbs_out/MILP/cap_sto_c_new
 - /urbs_out/MILP/cap_sto_p
 - /urbs_out/MILP/cap_sto_p_new
 - /urbs_out/MILP/costs
 - /urbs_out/MILP/dt
 - /urbs_out/MILP/e_co_buy
 - /urbs_out/MILP/e_co_sell
 - /urbs_out/MILP/e_pro_in
 - /urbs_out/MILP/e_pro_out
 - /urbs_out/MILP/e_sto_con
 - /urbs_out/MILP/e_sto_in
 - /urbs_out/MILP/e_s

In [28]:
# Path to the final inferred .h5 file (*_DmdAllocated_Features_Predicted.h5)
path_2_NEW = "/home/pedro/Linux-AntigravityProjects/SurroGrid-linux/Inference/3.Forward_Pass/0197_N2780244E4431460_80689_2_2_DmdAllocated_Features_Predicted.h5"

# Print the number of buildings in this grid and list available tables
with pd.HDFStore(path_2_NEW, mode="r") as store:
    df_bldgs_new = store["/raw_data/buildings"]
    n_res = int((df_bldgs_new["nonresidential_floor_area"] == 0).sum())
    n_nonres = int((df_bldgs_new["residential_floor_area"] == 0).sum())
    n_mixed = int(((df_bldgs_new["residential_floor_area"] != 0) & (df_bldgs_new["nonresidential_floor_area"] != 0)).sum())
    print(f"New (Inference) grid buildings: {len(df_bldgs_new)} total ({n_res} pure res, {n_nonres} pure non-res, {n_mixed} mixed)")

    keys = store.keys()
    print("Available tables in New file:")
    for k in keys:
        print(f" - {k}")


New (Inference) grid buildings: 125 total (118 pure res, 6 pure non-res, 1 mixed)
Available tables in New file:
 - /performance
 - /urbs_in/demand
 - /urbs_in/eff_factor
 - /urbs_in/process
 - /urbs_in/supim
 - /urbs_in/weather
 - /surrogate_out/demand_import
 - /surrogate_out/summary
 - /surrogate_in/features_raw
 - /surrogate_in/features_scaled
 - /surrogate_in/features_vmd
 - /raw_data/buildings
 - /raw_data/consumers
 - /raw_data/region
 - /raw_data/weather
 - /pwrflw/output/pre/demand_import
 - /pwrflw/output/pre/line_loads
 - /pwrflw/output/pre/vm
 - /pwrflw/input/demand_pre


# Comparison of the outputs

In [29]:
# Plot the hourly outputs of the surrogate together with the ground truth time series, superposed on an interactive zoomable plot.
# Includes +/-10% error band around the hourly ground truth, red markers where prediction exceeds +/-10%, peak load markers, and transformer sn_mva on |S|.
# Tip: Click-and-drag to zoom into any region, double-click to reset, use the bottom range slider, or click the preset zoom buttons / legend items.
import numpy as np
import plotly.graph_objects as go

# Load Ground Truth post-expansion powerflow and Surrogate predicted outputs
gt_post = pd.read_hdf(path_to_Reference, key="/pwrflw/output/post/demand_import").reset_index(drop=True)
pred_post = pd.read_hdf(path_2_NEW, key="/surrogate_out/demand_import").reset_index(drop=True)
summary_new = pd.read_hdf(path_2_NEW, key="/surrogate_out/summary")

gt_post["s_mva"] = np.sqrt(gt_post["p_mw"]**2 + gt_post["q_mvar"]**2)
sn_mva = float(summary_new["trafo_sn_mva"].iloc[0]) if "trafo_sn_mva" in summary_new.columns else np.nan

def plot_surrogate_vs_gt(s_gt, s_pred, title, ylabel, sn_limit=None, err_frac=0.10, zoom_hours=168):
    """Interactive hourly Plotly comparison of GT vs Surrogate with +/-10% band and peak markers."""
    gt = pd.Series(s_gt, dtype=float).reset_index(drop=True)
    pr = pd.Series(s_pred, dtype=float).reset_index(drop=True)
    hrs = np.arange(len(gt))

    # Symmetric +/-10% band around hourly GT (using abs(gt) so it remains valid for negative PV feed-in)
    upper = gt + err_frac * np.abs(gt)
    lower = gt - err_frac * np.abs(gt)
    outside_mask = (pr > upper) | (pr < lower)

    # Peak load indices and values
    gt_peak_hr = int(gt.idxmax())
    gt_peak_val = float(gt.iloc[gt_peak_hr])
    pr_peak_hr = int(pr.idxmax())
    pr_peak_val = float(pr.iloc[pr_peak_hr])
    peak_err_pct = (pr_peak_val - gt_peak_val) / abs(gt_peak_val) * 100.0 if gt_peak_val != 0 else np.nan

    # Metrics
    corr = np.corrcoef(gt.values, pr.values)[0, 1]
    mae = float(np.mean(np.abs(pr.values - gt.values)))
    nmae_pct = mae / float(np.mean(np.abs(gt.values))) * 100.0
    pct_outside = float(outside_mask.mean() * 100.0)

    print(f"=== {title} ===")
    print(f"  GT:   mean={gt.mean():+.4f}, min={gt.min():+.4f}, peak={gt_peak_val:+.4f} (hour {gt_peak_hr})")
    print(f"  Pred: mean={pr.mean():+.4f}, min={pr.min():+.4f}, peak={pr_peak_val:+.4f} (hour {pr_peak_hr}) | Peak err={peak_err_pct:+.2f}% (dt={pr_peak_hr - gt_peak_hr:+d} h)")
    print(f"  Pearson r={corr:.4f} | MAE={mae:.4f} | NMAE={nmae_pct:.2f}% | Outside +/-10% band: {pct_outside:.1f}% of hours")
    if sn_limit is not None and np.isfinite(sn_limit) and sn_limit > 0:
        gt_ovr = int((gt > sn_limit).sum())
        pr_ovr = int((pr > sn_limit).sum())
        print(f"  Trafo sn_mva={sn_limit:.3f} MVA | GT max loading={gt_peak_val/sn_limit*100:.1f}% ({gt_ovr} h >100%) | Pred max loading={pr_peak_val/sn_limit*100:.1f}% ({pr_ovr} h >100%)")

    # Zoom preset windows centered on GT peak
    wk_start = max(0, min(len(gt) - zoom_hours, gt_peak_hr - zoom_hours // 2))
    wk_end = wk_start + zoom_hours
    d3_start = max(0, min(len(gt) - 72, gt_peak_hr - 36))
    d3_end = d3_start + 72

    fig = go.Figure()

    # +/-10% shaded error band + dotted upper/lower boundaries around hourly GT
    fig.add_trace(go.Scatter(
        x=hrs, y=upper.values, mode="lines", name="GT +10%",
        line=dict(color="rgba(31, 119, 180, 0.50)", width=1.0, dash="dot"),
        legendgroup="err_band", showlegend=False,
        hovertemplate="+10%: %{y:.4f}<extra></extra>",
    ))
    fig.add_trace(go.Scatter(
        x=hrs, y=lower.values, mode="lines", name="GT +/-10% band",
        line=dict(color="rgba(31, 119, 180, 0.50)", width=1.0, dash="dot"),
        fill="tonexty", fillcolor="rgba(31, 119, 180, 0.14)",
        legendgroup="err_band",
        hovertemplate="-10%: %{y:.4f}<extra></extra>",
    ))

    # Hourly Ground Truth and Surrogate Prediction
    fig.add_trace(go.Scatter(
        x=hrs, y=gt.values, mode="lines", name="Ground Truth",
        line=dict(color="#1f77b4", width=1.5),
        hovertemplate="GT: %{y:.4f}<extra></extra>",
    ))
    fig.add_trace(go.Scatter(
        x=hrs, y=pr.values, mode="lines", name="Surrogate Pred",
        line=dict(color="#ff7f0e", width=1.4, dash="dash"),
        hovertemplate="Pred: %{y:.4f}<extra></extra>",
    ))

    # Highlight hours where Surrogate Pred exceeds +/-10% band
    if outside_mask.any():
        out_idx = hrs[outside_mask.values]
        fig.add_trace(go.Scatter(
            x=out_idx, y=pr.values[outside_mask.values], mode="markers",
            name="Pred outside +/-10%",
            marker=dict(color="red", size=5, opacity=0.75),
            hovertemplate="Outside +/-10%: %{y:.4f}<extra></extra>",
        ))

    # Peak load markers
    fig.add_trace(go.Scatter(
        x=[gt_peak_hr], y=[gt_peak_val], mode="markers",
        name=f"GT Peak ({gt_peak_val:.3f} @ h={gt_peak_hr})",
        marker=dict(color="navy", symbol="star", size=14, line=dict(color="white", width=0.8)),
        hovertemplate="GT Peak: %{y:.4f} (h=%{x})<extra></extra>",
    ))
    fig.add_trace(go.Scatter(
        x=[pr_peak_hr], y=[pr_peak_val], mode="markers",
        name=f"Pred Peak ({pr_peak_val:.3f} @ h={pr_peak_hr})",
        marker=dict(color="crimson", symbol="triangle-up", size=12, line=dict(color="white", width=0.8)),
        hovertemplate="Pred Peak: %{y:.4f} (h=%{x})<extra></extra>",
    ))

    # Optional transformer nominal rating line
    if sn_limit is not None and np.isfinite(sn_limit):
        fig.add_trace(go.Scatter(
            x=[0, len(gt) - 1], y=[sn_limit, sn_limit], mode="lines",
            name=f"Trafo sn_mva ({sn_limit:.2f})",
            line=dict(color="black", width=1.4, dash="dash"),
            hoverinfo="skip",
        ))

    fig.update_layout(
        title=dict(text=f"{title} — Hourly (Interactive Zoom)", x=0.01, xanchor="left"),
        height=560,
        template="plotly_white",
        hovermode="x unified",
        dragmode="zoom",
        margin=dict(l=65, r=25, t=95, b=55),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1.0, font=dict(size=10)),
        xaxis=dict(
            title="Hour of Year",
            range=[0, len(gt) - 1],
            rangeslider=dict(visible=True, thickness=0.09),
        ),
        yaxis=dict(title=ylabel, fixedrange=False),
        updatemenus=[
            dict(
                type="buttons",
                direction="right",
                x=0.0, y=1.13, xanchor="left", yanchor="top",
                showactive=True,
                buttons=[
                    dict(label="Full Year", method="relayout", args=[{"xaxis.range": [0, len(gt) - 1], "yaxis.autorange": True}]),
                    dict(label=f"Peak Week ({wk_start}–{wk_end})", method="relayout", args=[{"xaxis.range": [wk_start, wk_end], "yaxis.autorange": True}]),
                    dict(label=f"Peak 3 Days ({d3_start}–{d3_end})", method="relayout", args=[{"xaxis.range": [d3_start, d3_end], "yaxis.autorange": True}]),
                ],
            )
        ],
    )
    fig.show()

# 1. Active Power P [MW]
plot_surrogate_vs_gt(gt_post["p_mw"], pred_post["p_mw"], "Transformer Active Load P (Post-Expansion)", "Active Power [MW]")

# 2. Reactive Power Q [MVAr]
plot_surrogate_vs_gt(gt_post["q_mvar"], pred_post["q_mvar"], "Transformer Reactive Load Q (Post-Expansion)", "Reactive Power [MVAr]")

# 3. Apparent Power |S| [MVA] (with transformer nominal rating sn_mva)
plot_surrogate_vs_gt(gt_post["s_mva"], pred_post["s_mva"], "Transformer Apparent Load |S| (Post-Expansion)", "Apparent Power [MVA]", sn_limit=sn_mva)

# Show saved summary table from /surrogate_out/summary
display(summary_new)


=== Transformer Active Load P (Post-Expansion) ===
  GT:   mean=+0.1900, min=-0.3871, peak=+1.0028 (hour 606)
  Pred: mean=+0.1932, min=-0.3246, peak=+1.1545 (hour 1015) | Peak err=+15.13% (dt=+409 h)
  Pearson r=0.9905 | MAE=0.0225 | NMAE=9.54% | Outside +/-10% band: 47.4% of hours


=== Transformer Reactive Load Q (Post-Expansion) ===
  GT:   mean=+0.0601, min=+0.0004, peak=+0.2698 (hour 618)
  Pred: mean=+0.0612, min=-0.0003, peak=+0.2851 (hour 618) | Peak err=+5.64% (dt=+0 h)
  Pearson r=0.9919 | MAE=0.0046 | NMAE=7.69% | Outside +/-10% band: 44.5% of hours


=== Transformer Apparent Load |S| (Post-Expansion) ===
  GT:   mean=+0.2449, min=+0.0005, peak=+1.0289 (hour 606)
  Pred: mean=+0.2458, min=+0.0018, peak=+1.1853 (hour 1015) | Peak err=+15.20% (dt=+409 h)
  Pearson r=0.9868 | MAE=0.0225 | NMAE=9.19% | Outside +/-10% band: 46.2% of hours
  Trafo sn_mva=0.630 MVA | GT max loading=163.3% (441 h >100%) | Pred max loading=188.1% (467 h >100%)


,model_name,trafo_sn_mva,peak_p_import_mw,peak_p_feedin_mw,peak_s_mva,max_loading_pct,hours_overloaded_100pct,needs_replacement
0,baseline_alpha_peak_HPOfromOct17th,0.63,1.154485,-0.324557,1.185285,188.140472,467,True


# Comparison of the generated cheap ground truth

In [30]:
# Helper function for interactive hourly comparative plotting (no daily averaging)
# Tip: Click-and-drag to zoom, double-click to reset, or use the bottom range slider / quick zoom buttons.
import numpy as np
import plotly.graph_objects as go

def plot_feature_comparison(s_ref, s_new, title, ylabel, zoom_start=240, zoom_hours=168, is_power_kw=False):
    """Interactive hourly Plotly comparison of Reference (GT) vs New (Inference)."""
    s_ref = pd.Series(s_ref, dtype=float).reset_index(drop=True)
    s_new = pd.Series(s_new, dtype=float).reset_index(drop=True)
    hrs = np.arange(len(s_ref))
    corr = np.corrcoef(s_ref.values, s_new.values)[0, 1]

    # Summary metrics
    print(f"{title}")
    print(f"  Reference: min={s_ref.min():.4f}, mean={s_ref.mean():.4f}, max={s_ref.max():.4f}, NaNs={s_ref.isna().sum()}")
    print(f"  New:       min={s_new.min():.4f}, mean={s_new.mean():.4f}, max={s_new.max():.4f}, NaNs={s_new.isna().sum()}")
    if is_power_kw:
        print(f"  Annual sum: Reference={s_ref.sum()/1000:.2f} MWh | New={s_new.sum()/1000:.2f} MWh ({(s_new.sum()/s_ref.sum()-1)*100:+.2f}%)")
    print(f"  Pearson r: {corr:.4f}")

    zoom_end = min(len(s_ref) - 1, zoom_start + zoom_hours)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=hrs, y=s_ref.values, mode="lines", name="Reference (GT)",
        line=dict(color="#1f77b4", width=1.5),
        hovertemplate="Ref: %{y:.4f}<extra></extra>",
    ))
    fig.add_trace(go.Scatter(
        x=hrs, y=s_new.values, mode="lines", name="New (Inference)",
        line=dict(color="#ff7f0e", width=1.4, dash="dash"),
        hovertemplate="New: %{y:.4f}<extra></extra>",
    ))

    fig.update_layout(
        title=dict(text=f"{title} — Hourly (Interactive Zoom)", x=0.01, xanchor="left"),
        height=480,
        template="plotly_white",
        hovermode="x unified",
        dragmode="zoom",
        margin=dict(l=65, r=25, t=90, b=55),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1.0, font=dict(size=10)),
        xaxis=dict(
            title="Hour of Year",
            range=[0, len(s_ref) - 1],
            rangeslider=dict(visible=True, thickness=0.09),
        ),
        yaxis=dict(title=ylabel, fixedrange=False),
        updatemenus=[
            dict(
                type="buttons",
                direction="right",
                x=0.0, y=1.14, xanchor="left", yanchor="top",
                showactive=True,
                buttons=[
                    dict(label="Full Year", method="relayout", args=[{"xaxis.range": [0, len(s_ref) - 1], "yaxis.autorange": True}]),
                    dict(label=f"1-Week Zoom ({zoom_start}–{zoom_end})", method="relayout", args=[{"xaxis.range": [zoom_start, zoom_end], "yaxis.autorange": True}]),
                ],
            )
        ],
    )
    fig.show()


In [31]:
# Display the Electrical load time series on the transformer before urbs for both the 
# reference and the new .h5 files
elec_ref = pd.read_hdf(path_to_Reference, key="/pwrflw/output/pre/demand_import")["p_mw"]
elec_new = pd.read_hdf(path_2_NEW, key="/pwrflw/output/pre/demand_import")["p_mw"]

plot_feature_comparison(
    elec_ref, elec_new,
    title="Pre-URBS Transformer Active Load (demand_net_active_pre)",
    ylabel="Active Power [MW]",
    zoom_start=240,  # mid-January week
)


Pre-URBS Transformer Active Load (demand_net_active_pre)
  Reference: min=0.0575, mean=0.1282, max=0.2641, NaNs=0
  New:       min=0.0550, mean=0.1332, max=0.2885, NaNs=0
  Pearson r: 0.9825


In [32]:
# Display the aggregated DHW Time series for both the 
# reference and the new .h5 files
dmd_ref = pd.read_hdf(path_to_Reference, key="/urbs_in/demand")
dmd_new = pd.read_hdf(path_2_NEW, key="/urbs_in/demand")

dhw_ref = dmd_ref.loc[:, dmd_ref.columns.get_level_values(1).str.startswith("water_heat")].sum(axis=1)
dhw_new = dmd_new.loc[:, dmd_new.columns.get_level_values(1).str.startswith("water_heat")].sum(axis=1)

plot_feature_comparison(
    dhw_ref, dhw_new,
    title="Aggregated DHW Demand (heat_water)",
    ylabel="Thermal Power [kW]",
    zoom_start=240,
    is_power_kw=True,
)


Aggregated DHW Demand (heat_water)
  Reference: min=1.8258, mean=53.6857, max=386.1566, NaNs=0
  New:       min=1.1871, mean=53.3164, max=396.4302, NaNs=0
  Annual sum: Reference=470.29 MWh | New=467.05 MWh (-0.69%)
  Pearson r: 0.9950


In [33]:
# Display the aggregated Space Heat Time series for both the 
# reference and the new .h5 files
sh_ref = dmd_ref.loc[:, dmd_ref.columns.get_level_values(1).str.startswith("space_heat")].sum(axis=1)
sh_new = dmd_new.loc[:, dmd_new.columns.get_level_values(1).str.startswith("space_heat")].sum(axis=1)

plot_feature_comparison(
    sh_ref, sh_new,
    title="Aggregated Space Heat Demand (heat_space)",
    ylabel="Thermal Power [kW]",
    zoom_start=240,
    is_power_kw=True,
)


Aggregated Space Heat Demand (heat_space)
  Reference: min=0.0000, mean=433.1829, max=2021.5139, NaNs=0
  New:       min=0.0000, mean=431.9490, max=2016.8056, NaNs=0
  Annual sum: Reference=3794.68 MWh | New=3783.87 MWh (-0.28%)
  Pearson r: 1.0000


In [34]:
# Display the Grid's COP (average for every building, weighted by the building's total heat demand)
# Time serie for both the reference and the new .h5 files
def compute_weighted_cop(h5_path):
    """Exact get_heat_and_cop logic from GridForecast/0_preprocessing/0.extract_ML_data.ipynb."""
    with pd.HDFStore(h5_path, mode="r") as store:
        demand = store["urbs_in/demand"]
        eff_factor = store["urbs_in/eff_factor"]

    space_heat = demand.loc[:, demand.columns.get_level_values(1).str.startswith("space_heat")].droplevel(1, axis=1)
    water_heat = demand.loc[:, demand.columns.get_level_values(1).str.startswith("water_heat")].droplevel(1, axis=1)
    hp_cop = eff_factor.loc[:, eff_factor.columns.get_level_values(1).str.startswith("heatpump_air")].droplevel(1, axis=1)

    total_heat_cols = space_heat + water_heat
    denom = total_heat_cols.sum(axis=1)
    weighted_sum = hp_cop.multiply(total_heat_cols).sum(axis=1)

    weighted_cop = pd.Series(np.nan, index=denom.index, dtype=float)
    nonzero = denom != 0
    weighted_cop.loc[nonzero] = weighted_sum.loc[nonzero] / denom.loc[nonzero]
    weighted_cop.loc[~nonzero] = hp_cop.mean(axis=1).loc[~nonzero]
    return weighted_cop

cop_ref = compute_weighted_cop(path_to_Reference)
cop_new = compute_weighted_cop(path_2_NEW)

plot_feature_comparison(
    cop_ref, cop_new,
    title="Grid Weighted Average Heat Pump COP (cop_avg)",
    ylabel="COP [-]",
    zoom_start=240,
)


Grid Weighted Average Heat Pump COP (cop_avg)
  Reference: min=2.3018, mean=3.7499, max=4.8284, NaNs=0
  New:       min=2.3227, mean=3.7627, max=4.8326, NaNs=0
  Pearson r: 0.9991


In [35]:
# Display the Expected PV Production (PV_prod_expected) time series for both files
def compute_pv_prod_expected(h5_path, fraction_of_total=0.6766):
    """Exact create_PV_prod_up_to logic from GridForecast/0_preprocessing/0.extract_ML_data.ipynb."""
    with pd.HDFStore(h5_path, mode="r") as store:
        supim = store["urbs_in/supim"]
        processes = store["urbs_in/process"]

    pv_100 = processes[processes["Process"].str.startswith("Rooftop PV")].set_index(["Site", "Process"])["cap-up"].to_frame().T
    cap_map = {
        (site, cap_proc.replace("Rooftop PV", "solar")): pv_100[(site, cap_proc)].iloc[0]
        for site, cap_proc in pv_100.columns
    }
    cap_series = pd.Series([cap_map.get((site, proc), pd.NA) for site, proc in supim.columns], index=supim.columns)
    df_scaled = supim * cap_series

    target_energy = df_scaled.sum().sum() * fraction_of_total
    order = supim.sum(axis=0).sort_values(ascending=False).index
    energy_sums = df_scaled.sum(axis=0)

    selected_ts = []
    total_energy = 0.0
    for col in order:
        col_energy = energy_sums[col]
        if total_energy + col_energy <= target_energy:
            selected_ts.append(df_scaled[col])
            total_energy += col_energy
        else:
            frac = (target_energy - total_energy) / col_energy
            selected_ts.append(df_scaled[col] * frac)
            break

    return pd.concat(selected_ts, axis=1).sum(axis=1)

pv_ref = compute_pv_prod_expected(path_to_Reference)
pv_new = compute_pv_prod_expected(path_2_NEW)

plot_feature_comparison(
    pv_ref, pv_new,
    title="Expected PV Production (PV_prod_expected)",
    ylabel="Power [kW]",
    zoom_start=4320,  # late-June summer week
    is_power_kw=True,
)


Expected PV Production (PV_prod_expected)
  Reference: min=0.0000, mean=150.2881, max=882.6389, NaNs=0
  New:       min=0.0000, mean=146.4398, max=868.6719, NaNs=0
  Annual sum: Reference=1316.52 MWh | New=1282.81 MWh (-2.56%)
  Pearson r: 0.9998


In [36]:
# Check if there are any remaining active MV-level buildings in the .h5 files
# Thresholds (consistent with disconnect_MV_buildings.py):
#   - Pure non-residential (residential_floor_area == 0): peak electricity > 100 kW
#   - Pure residential / mixed (residential_floor_area > 0): peak electricity > 250 kW
import io
import json
import h5py

def check_mv_buildings(h5_path, label, nonres_thresh_kw=100.0, res_thresh_kw=250.0):
    df_buildings = pd.read_hdf(h5_path, key="/raw_data/buildings")
    df_demand = pd.read_hdf(h5_path, key="/urbs_in/demand")
    df_elec = df_demand.loc[:, df_demand.columns.get_level_values(1) == "electricity"]
    peak_by_bus = {int(float(col[0])): float(df_elec[col].max()) for col in df_elec.columns}

    # Read load table from raw_data/net JSON to inspect in_service status
    with h5py.File(h5_path, "r") as f:
        net_json = f["raw_data/net"][()]
        if isinstance(net_json, bytes):
            net_json = net_json.decode("utf-8")
        net_data = json.loads(net_json)
    load_info = net_data["_object"]["load"]
    df_load = pd.read_json(io.StringIO(load_info["_object"]), orient=load_info["orient"])
    in_service_by_bus = df_load.groupby(df_load["bus"].astype(int))["in_service"].any().to_dict()

    rows = []
    for _, row in df_buildings.iterrows():
        bus = int(float(row["bus"]))
        res_area = 0.0 if pd.isna(row.get("residential_floor_area")) else float(row.get("residential_floor_area"))
        thresh = nonres_thresh_kw if res_area == 0.0 else res_thresh_kw
        peak_kw = peak_by_bus.get(bus, 0.0)
        in_srv = bool(in_service_by_bus.get(bus, True))
        rows.append({
            "bus": bus,
            "use": row.get("use"),
            "res_area_m2": res_area,
            "peak_elec_kw": peak_kw,
            "threshold_kw": thresh,
            "exceeds_threshold": peak_kw > thresh,
            "in_service": in_srv,
        })

    df_check = pd.DataFrame(rows)
    disconnected = df_check[~df_check["in_service"]]
    active_mv_violations = df_check[df_check["exceeds_threshold"] & df_check["in_service"]]

    print(f"=== MV Building Check: {label} ===")
    print(f"  Total buildings:                   {len(df_check)}")
    print(f"  Disconnected (in_service=False):   {len(disconnected)} (buses: {disconnected['bus'].tolist()})")
    print(f"  Active buildings above threshold:  {len(active_mv_violations)}")
    print(f"  Max active building peak (kW):     {df_check.loc[df_check['in_service'], 'peak_elec_kw'].max():.2f} kW")
    if not active_mv_violations.empty:
        print("  WARNING — Remaining active MV buildings found:")
        display(active_mv_violations)
    return df_check.sort_values("peak_elec_kw", ascending=False)

df_mv_ref = check_mv_buildings(path_to_Reference, "Reference (GT)")
print()
df_mv_new = check_mv_buildings(path_2_NEW, "New (Inference)")
print("\nTop 5 highest-peak buildings in New (Inference):")
display(df_mv_new.head(5))


=== MV Building Check: Reference (GT) ===
  Total buildings:                   125
  Disconnected (in_service=False):   0 (buses: [])
  Active buildings above threshold:  0
  Max active building peak (kW):     35.13 kW

=== MV Building Check: New (Inference) ===
  Total buildings:                   125
  Disconnected (in_service=False):   0 (buses: [])
  Active buildings above threshold:  0
  Max active building peak (kW):     37.64 kW

Top 5 highest-peak buildings in New (Inference):


,bus,use,res_area_m2,peak_elec_kw,threshold_kw,exceeds_threshold,in_service
44,169,Residential,5058.414,37.641855,250.0,False,True
42,167,Residential,5254.830,32.601130,250.0,False,True
52,177,Residential,872.336,22.618098,250.0,False,True
43,168,Residential,966.804,18.478901,250.0,False,True
56,181,Residential,631.420,16.914000,250.0,False,True
